### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)
        * [Positive vs. Negative](#512-positive-vs-negative)




### 1. Environment Setup

##### 1.1 Library Imports

In [ ]:
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [4]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [5]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are optimized using Optuna with 5-fold StratifiedGroupKFold on the full dataset. The resulting best parameters are fixed and used for final evaluation with LOSO. For comparison, performance is also assessed using standard 10-fold cross-validation.

##### General Functions for Training

In [10]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def xgb_hyperparameter_training_loso(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [11]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def xgb_hyperparameter_training_10f(X, y, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [12]:
def xgb_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #Per fold class weighting
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg/pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [13]:
def xgb_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)
        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg / pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        # importance = model.feature_importances_ 
        # print(len(importance))

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [14]:
#Hyperparameter tuning
xgb_en_params_loso = xgb_hyperparameter_training_loso(X_en, y_en, groups_en, "XGBoost", "Emotional vs. Neutral")

[I 2026-03-25 15:18:36,587] A new study created in memory with name: no-name-07798521-2844-49d1-989a-1ac90c47a0f4


=== Hyperparameter Tuning - XGBoost (Emotional vs. Neutral) ===


[I 2026-03-25 15:18:38,711] Trial 0 finished with value: 0.4822826590701448 and parameters: {'n_estimators': 499, 'max_depth': 4, 'learning_rate': 0.2988653806906803, 'subsample': 0.7971707025459515, 'colsample_bytree': 0.7138474014117419, 'min_child_weight': 6, 'gamma': 3.858983587029829}. Best is trial 0 with value: 0.4822826590701448.
[I 2026-03-25 15:18:40,654] Trial 1 finished with value: 0.4803416628852368 and parameters: {'n_estimators': 180, 'max_depth': 7, 'learning_rate': 0.1045448777578841, 'subsample': 0.8564443254137115, 'colsample_bytree': 0.7520101514006271, 'min_child_weight': 1, 'gamma': 2.5778787714189733}. Best is trial 0 with value: 0.4822826590701448.
[I 2026-03-25 15:18:47,083] Trial 2 finished with value: 0.46835012209868543 and parameters: {'n_estimators': 409, 'max_depth': 3, 'learning_rate': 0.03625328942535395, 'subsample': 0.8140427098147297, 'colsample_bytree': 0.999591499837405, 'min_child_weight': 1, 'gamma': 2.230309955439061}. Best is trial 0 with value

Best params: {'n_estimators': 530, 'max_depth': 3, 'learning_rate': 0.2726748229308611, 'subsample': 0.8480492910959272, 'colsample_bytree': 0.9740426358784037, 'min_child_weight': 9, 'gamma': 1.7944407127037547}
Best CV F1: 0.5070


In [15]:
#Hyperparameter tuning
xgb_en_params_10f = xgb_hyperparameter_training_10f(X_en, y_en, "XGBoost", "Emotional vs. Neutral")

[I 2026-03-25 15:22:25,991] A new study created in memory with name: no-name-c90fe3e2-64c0-4aa4-a40f-35bee25a80fd


=== Hyperparameter Tuning - XGBoost (Emotional vs. Neutral) ===


[I 2026-03-25 15:22:28,470] Trial 0 finished with value: 0.6764392798448966 and parameters: {'n_estimators': 302, 'max_depth': 5, 'learning_rate': 0.1712551172327458, 'subsample': 0.6258253720543554, 'colsample_bytree': 0.6647975877227844, 'min_child_weight': 9, 'gamma': 3.7037977165245146}. Best is trial 0 with value: 0.6764392798448966.
[I 2026-03-25 15:22:34,063] Trial 1 finished with value: 0.6873110772176111 and parameters: {'n_estimators': 213, 'max_depth': 7, 'learning_rate': 0.018852821446601574, 'subsample': 0.6732285043569226, 'colsample_bytree': 0.8627360645452999, 'min_child_weight': 5, 'gamma': 3.2612753842700997}. Best is trial 1 with value: 0.6873110772176111.
[I 2026-03-25 15:22:38,236] Trial 2 finished with value: 0.671250124454872 and parameters: {'n_estimators': 360, 'max_depth': 4, 'learning_rate': 0.020534349204279476, 'subsample': 0.795912667859033, 'colsample_bytree': 0.9028990325470566, 'min_child_weight': 10, 'gamma': 4.096109378304196}. Best is trial 1 with va

Best params: {'n_estimators': 411, 'max_depth': 6, 'learning_rate': 0.05451894861068764, 'subsample': 0.7483775734232367, 'colsample_bytree': 0.6357783505641547, 'min_child_weight': 2, 'gamma': 1.3597307815044044}
Best CV F1: 0.7040


In [16]:
#LOSO Evaluation
xgb_en_loso = xgb_loso_loop(X_en, y_en, groups_en, xgb_en_params_loso, "XGBoost", "Emotional vs. Neutral")


=== LOSO - XGBoost (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.4451 | Accuracy: 0.5351 | F1: 0.4031 | AUROC: 0.4491
Subject 003 | Balanced Accuracy: 0.4493 | Accuracy: 0.4450 | F1: 0.4439 | AUROC: 0.4262
Subject 004 | Balanced Accuracy: 0.4733 | Accuracy: 0.4490 | F1: 0.4445 | AUROC: 0.4743
Subject 005 | Balanced Accuracy: 0.5484 | Accuracy: 0.5992 | F1: 0.5486 | AUROC: 0.5571
Subject 007 | Balanced Accuracy: 0.5188 | Accuracy: 0.5675 | F1: 0.4691 | AUROC: 0.5148
Subject 012 | Balanced Accuracy: 0.7008 | Accuracy: 0.7263 | F1: 0.6026 | AUROC: 0.7349
Subject 013 | Balanced Accuracy: 0.4821 | Accuracy: 0.4615 | F1: 0.4313 | AUROC: 0.3571
Subject 015 | Balanced Accuracy: 0.4050 | Accuracy: 0.4015 | F1: 0.4015 | AUROC: 0.4233
Subject 016 | Balanced Accuracy: 0.4740 | Accuracy: 0.5681 | F1: 0.4428 | AUROC: 0.5144
Subject 017 | Balanced Accuracy: 0.4602 | Accuracy: 0.3052 | F1: 0.3017 | AUROC: 0.3886
Subject 020 | Balanced Accuracy: 0.4684 | Accuracy: 0.3619 | F1: 0.3202 

In [17]:
#10Fold CV
xgb_en_10f = xgb_ten_fold_cv_loop(X_en, y_en, xgb_en_params_10f, "XGBoost", "Emotional vs. Neutral")


=== 10-Fold CV - XGBoost (Emotional vs. Neutral) ===
Accuracy:          0.7071 ± 0.0109
F1:                0.7053 ± 0.0106
Balanced Accuracy: 0.7055 ± 0.0102
AUROC:             0.7688 ± 0.0090


##### 5.1.2 Positive vs. Negative

In [18]:
#Hyperparameter tuning
xgb_pn_params_loso = xgb_hyperparameter_training_loso(X_pn, y_pn, groups_pn, "XGBoost", "Positive vs. Negative")

[I 2026-03-25 15:28:28,141] A new study created in memory with name: no-name-0ece0fef-f7c4-4791-8b00-df18e28d81b0


=== Hyperparameter Tuning - XGBoost (Positive vs. Negative) ===


[I 2026-03-25 15:28:33,542] Trial 0 finished with value: 0.5286066268678162 and parameters: {'n_estimators': 471, 'max_depth': 5, 'learning_rate': 0.014355233666079936, 'subsample': 0.8362243573618626, 'colsample_bytree': 0.6817825875240787, 'min_child_weight': 10, 'gamma': 0.7025694013403078}. Best is trial 0 with value: 0.5286066268678162.
[I 2026-03-25 15:28:35,129] Trial 1 finished with value: 0.5216745788196688 and parameters: {'n_estimators': 294, 'max_depth': 3, 'learning_rate': 0.273390793332756, 'subsample': 0.6998741784737862, 'colsample_bytree': 0.6817401469761669, 'min_child_weight': 4, 'gamma': 2.4694312967973033}. Best is trial 0 with value: 0.5286066268678162.
[I 2026-03-25 15:28:37,296] Trial 2 finished with value: 0.5182432460270244 and parameters: {'n_estimators': 309, 'max_depth': 3, 'learning_rate': 0.12837668559897003, 'subsample': 0.997625224621245, 'colsample_bytree': 0.7152972014282418, 'min_child_weight': 2, 'gamma': 0.4810560537166031}. Best is trial 0 with va

Best params: {'n_estimators': 103, 'max_depth': 7, 'learning_rate': 0.025486928572465935, 'subsample': 0.6475048098930399, 'colsample_bytree': 0.608808148371969, 'min_child_weight': 8, 'gamma': 2.64702467249101}
Best CV F1: 0.5435


In [19]:
#Hyperparameter tuning
xgb_pn_params_10f = xgb_hyperparameter_training_10f(X_pn, y_pn, "XGBoost", "Positive vs. Negative")

[I 2026-03-25 15:31:23,511] A new study created in memory with name: no-name-6e69c4cb-ece0-4254-86d4-5876258a6a1e


=== Hyperparameter Tuning - XGBoost (Positive vs. Negative) ===


[I 2026-03-25 15:31:26,826] Trial 0 finished with value: 0.6654891476590705 and parameters: {'n_estimators': 325, 'max_depth': 4, 'learning_rate': 0.019308082866789283, 'subsample': 0.867395830364148, 'colsample_bytree': 0.815394602384188, 'min_child_weight': 1, 'gamma': 3.141016605414701}. Best is trial 0 with value: 0.6654891476590705.
[I 2026-03-25 15:31:29,554] Trial 1 finished with value: 0.6314156426855895 and parameters: {'n_estimators': 397, 'max_depth': 3, 'learning_rate': 0.010865013751964407, 'subsample': 0.9005126166264167, 'colsample_bytree': 0.6089592526064248, 'min_child_weight': 9, 'gamma': 1.940911103542073}. Best is trial 0 with value: 0.6654891476590705.
[I 2026-03-25 15:31:31,640] Trial 2 finished with value: 0.6562821599112414 and parameters: {'n_estimators': 595, 'max_depth': 5, 'learning_rate': 0.21042938895380003, 'subsample': 0.8745189664422092, 'colsample_bytree': 0.6840484143120328, 'min_child_weight': 10, 'gamma': 2.8893602858116862}. Best is trial 0 with va

Best params: {'n_estimators': 175, 'max_depth': 8, 'learning_rate': 0.027177753336299713, 'subsample': 0.9232506300648756, 'colsample_bytree': 0.6791317599235943, 'min_child_weight': 1, 'gamma': 1.784112433710707}
Best CV F1: 0.6887


In [20]:
#LOSO Evaluation
xgb_pn_loso = xgb_loso_loop(X_pn, y_pn, groups_pn, xgb_pn_params_loso, "XGBoost", "Positive vs. Negative")


=== LOSO - XGBoost (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.5559 | Accuracy: 0.5689 | F1: 0.5440 | AUROC: 0.6039
Subject 003 | Balanced Accuracy: 0.4214 | Accuracy: 0.5189 | F1: 0.4187 | AUROC: 0.3450
Subject 004 | Balanced Accuracy: 0.6284 | Accuracy: 0.6241 | F1: 0.6204 | AUROC: 0.6934
Subject 005 | Balanced Accuracy: 0.4680 | Accuracy: 0.4713 | F1: 0.4627 | AUROC: 0.4181
Subject 007 | Balanced Accuracy: 0.5029 | Accuracy: 0.5035 | F1: 0.5028 | AUROC: 0.5024
Subject 015 | Balanced Accuracy: 0.5027 | Accuracy: 0.8231 | F1: 0.4868 | AUROC: 0.2167
Subject 017 | Balanced Accuracy: 0.6071 | Accuracy: 0.6667 | F1: 0.5846 | AUROC: 0.6964
Subject 021 | Balanced Accuracy: 0.5352 | Accuracy: 0.7981 | F1: 0.5367 | AUROC: 0.5691
Subject 022 | Balanced Accuracy: 0.5559 | Accuracy: 0.5474 | F1: 0.5165 | AUROC: 0.6486
Subject 023 | Balanced Accuracy: 0.4099 | Accuracy: 0.6200 | F1: 0.4062 | AUROC: 0.3858
Subject 024 | Balanced Accuracy: 0.4389 | Accuracy: 0.4468 | F1: 0.4383 

In [21]:
#10Fold CV
xgb_pn_10f = xgb_ten_fold_cv_loop(X_pn, y_pn, xgb_pn_params_10f, "XGBoost", "Positive vs. Negative")


=== 10-Fold CV - XGBoost (Positive vs. Negative) ===
Accuracy:          0.6971 ± 0.0249
F1:                0.6917 ± 0.0248
Balanced Accuracy: 0.6957 ± 0.0246
AUROC:             0.7625 ± 0.0270


#### KNN

In [22]:
def subject_normalize(X, groups):
    X_norm = X.copy()
    for subj in np.unique(groups):
        mask = groups == subj
        scaler = StandardScaler()
        X_norm[mask] = scaler.fit_transform(X[mask])
    return X_norm

In [23]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def knn_hyperparameter_training_loso(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            groups_train = groups[train_idx]
            groups_val   = groups[val_idx]

            X_train_norm = subject_normalize(X_train, groups_train)
            X_val_norm   = subject_normalize(X_val, groups_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train_norm, y_train)
            preds = model.predict(X_val_norm)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    return best_params

In [24]:
def knn_hyperparameter_training_10f(X, y, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning (10-Fold) - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")

    return study.best_params

In [25]:
def knn_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        groups_train = groups[train_idx]
        groups_test  = groups[test_idx]

        X_train_norm = subject_normalize(X_train, groups_train)
        X_test_norm  = subject_normalize(X_test, groups_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train_norm, y_train)
        preds = model.predict(X_test_norm)
        proba = model.predict_proba(X_test_norm)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [26]:
def knn_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)


        model = KNeighborsClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

In [27]:
#Hyperparameter tuning
knn_en_params_loso = knn_hyperparameter_training_loso(X_en, y_en, groups_en, "KNN", "Emotional vs. Neutral")

[I 2026-03-25 15:34:48,566] A new study created in memory with name: no-name-ead06582-2943-434f-bc29-3b4501cdd3b0


=== Hyperparameter Tuning - KNN (Emotional vs. Neutral) ===


[I 2026-03-25 15:34:49,421] Trial 0 finished with value: 0.49470331659923134 and parameters: {'n_neighbors': 4, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 17}. Best is trial 0 with value: 0.49470331659923134.
[I 2026-03-25 15:34:49,658] Trial 1 finished with value: 0.479158837354648 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 23}. Best is trial 0 with value: 0.49470331659923134.
[I 2026-03-25 15:34:53,020] Trial 2 finished with value: 0.4975525863597158 and parameters: {'n_neighbors': 9, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 21}. Best is trial 2 with value: 0.4975525863597158.
[I 2026-03-25 15:34:56,370] Trial 3 finished with value: 0.4975390460261548 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 31}. Best is trial 2 with value: 0.4975525863597158.
[I 2026-03-25 15:34:56,594] Trial 4 finished with value: 0.47739381966668154 and parameters: {'n_neighbors': 27, 

Best params: {'n_neighbors': 21, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 37}
Best CV F1: 0.5044


In [28]:
#Hyperparameter tuning
knn_en_params_10f = knn_hyperparameter_training_10f(X_en, y_en, "KNN", "Emotional vs. Neutral")

[I 2026-03-25 15:36:28,509] A new study created in memory with name: no-name-289c9c25-0a6f-44ac-8be3-284ed067f0f6


=== Hyperparameter Tuning (10-Fold) - KNN (Emotional vs. Neutral) ===


[I 2026-03-25 15:36:28,962] Trial 0 finished with value: 0.6448120899764991 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 11}. Best is trial 0 with value: 0.6448120899764991.
[I 2026-03-25 15:36:29,061] Trial 1 finished with value: 0.6176911239167484 and parameters: {'n_neighbors': 3, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 19}. Best is trial 0 with value: 0.6448120899764991.
[I 2026-03-25 15:36:32,171] Trial 2 finished with value: 0.6156888645331277 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 31}. Best is trial 0 with value: 0.6448120899764991.
[I 2026-03-25 15:36:32,313] Trial 3 finished with value: 0.6237831367387383 and parameters: {'n_neighbors': 18, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 46}. Best is trial 0 with value: 0.6448120899764991.
[I 2026-03-25 15:36:32,753] Trial 4 finished with value: 0.6526250418540502 and parameters: {'n_neighbors': 2

Best params: {'n_neighbors': 16, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 29}
Best CV F1: 0.6580


In [29]:
#LOSO Evaluation
knn_en_loso = knn_loso_loop(X_en, y_en, groups_en, knn_en_params_loso, "KNN", "Emotional vs. Neutral")


=== LOSO - KNN (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.5246 | Accuracy: 0.5892 | F1: 0.4503 | AUROC: 0.4607
Subject 003 | Balanced Accuracy: 0.4786 | Accuracy: 0.4850 | F1: 0.4754 | AUROC: 0.4540
Subject 004 | Balanced Accuracy: 0.4911 | Accuracy: 0.4898 | F1: 0.4876 | AUROC: 0.4854
Subject 005 | Balanced Accuracy: 0.4934 | Accuracy: 0.4835 | F1: 0.4779 | AUROC: 0.5091
Subject 007 | Balanced Accuracy: 0.4874 | Accuracy: 0.4697 | F1: 0.4120 | AUROC: 0.4829
Subject 012 | Balanced Accuracy: 0.5151 | Accuracy: 0.5263 | F1: 0.4361 | AUROC: 0.4413
Subject 013 | Balanced Accuracy: 0.6905 | Accuracy: 0.6923 | F1: 0.6905 | AUROC: 0.6071
Subject 015 | Balanced Accuracy: 0.4701 | Accuracy: 0.4796 | F1: 0.4684 | AUROC: 0.4485
Subject 016 | Balanced Accuracy: 0.4043 | Accuracy: 0.4225 | F1: 0.3570 | AUROC: 0.3265
Subject 017 | Balanced Accuracy: 0.4800 | Accuracy: 0.3380 | F1: 0.3309 | AUROC: 0.4129
Subject 020 | Balanced Accuracy: 0.5921 | Accuracy: 0.3429 | F1: 0.3230 | AU

In [30]:
#10Fold CV
knn_en_10f = knn_ten_fold_cv_loop(X_en, y_en, knn_en_params_10f, "KNN", "Emotional vs. Neutral")


=== 10-Fold CV - KNN (Emotional vs. Neutral) ===
Accuracy:          0.6641 ± 0.0204
F1:                0.6587 ± 0.0212
Balanced Accuracy: 0.6584 ± 0.0209
AUROC:             0.7168 ± 0.0163


In [31]:
#Hyperparameter tuning
knn_pn_params_loso = knn_hyperparameter_training_loso(X_pn, y_pn, groups_pn, "KNN", "Positive vs. Negative")

[I 2026-03-25 15:37:12,506] A new study created in memory with name: no-name-b7b3e999-7276-414d-9a48-93d434664444


=== Hyperparameter Tuning - KNN (Positive vs. Negative) ===


[I 2026-03-25 15:37:12,774] Trial 0 finished with value: 0.4893575615950926 and parameters: {'n_neighbors': 26, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 16}. Best is trial 0 with value: 0.4893575615950926.
[I 2026-03-25 15:37:13,000] Trial 1 finished with value: 0.48422110767062154 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 33}. Best is trial 0 with value: 0.4893575615950926.
[I 2026-03-25 15:37:14,033] Trial 2 finished with value: 0.4804677885260791 and parameters: {'n_neighbors': 9, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 47}. Best is trial 0 with value: 0.4893575615950926.
[I 2026-03-25 15:37:14,215] Trial 3 finished with value: 0.5027113370634406 and parameters: {'n_neighbors': 13, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 27}. Best is trial 3 with value: 0.5027113370634406.
[I 2026-03-25 15:37:14,390] Trial 4 finished with value: 0.49521141780201045 and parameters: {'n_neighbors': 4

Best params: {'n_neighbors': 12, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 43}
Best CV F1: 0.5160


In [32]:
#Hyperparameter tuning
knn_pn_params_10f = knn_hyperparameter_training_10f(X_pn, y_pn, "KNN", "Positive vs. Negative")

[I 2026-03-25 15:37:27,518] A new study created in memory with name: no-name-5953adfc-4a9c-41f8-a201-7c4619486fb3


=== Hyperparameter Tuning (10-Fold) - KNN (Positive vs. Negative) ===


[I 2026-03-25 15:37:27,734] Trial 0 finished with value: 0.634720670318529 and parameters: {'n_neighbors': 22, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 18}. Best is trial 0 with value: 0.634720670318529.
[I 2026-03-25 15:37:28,647] Trial 1 finished with value: 0.5695791906147211 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 35}. Best is trial 0 with value: 0.634720670318529.
[I 2026-03-25 15:37:29,557] Trial 2 finished with value: 0.5538816921352957 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 22}. Best is trial 0 with value: 0.634720670318529.
[I 2026-03-25 15:37:30,366] Trial 3 finished with value: 0.5910426145098441 and parameters: {'n_neighbors': 1, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 41}. Best is trial 0 with value: 0.634720670318529.
[I 2026-03-25 15:37:30,572] Trial 4 finished with value: 0.6299802648854401 and parameters: {'n_neighbors': 12, 'weights

Best params: {'n_neighbors': 24, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 48}
Best CV F1: 0.6364


In [33]:
#LOSO Evaluation
knn_pn_loso = knn_loso_loop(X_pn, y_pn, groups_pn, knn_pn_params_loso, "KNN", "Positive vs. Negative")


=== LOSO - KNN (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.5910 | Accuracy: 0.5868 | F1: 0.5866 | AUROC: 0.6004
Subject 003 | Balanced Accuracy: 0.5079 | Accuracy: 0.5377 | F1: 0.5003 | AUROC: 0.5100
Subject 004 | Balanced Accuracy: 0.5009 | Accuracy: 0.5035 | F1: 0.4993 | AUROC: 0.5154
Subject 005 | Balanced Accuracy: 0.4848 | Accuracy: 0.5287 | F1: 0.4825 | AUROC: 0.5407
Subject 007 | Balanced Accuracy: 0.4809 | Accuracy: 0.4896 | F1: 0.4676 | AUROC: 0.4755
Subject 015 | Balanced Accuracy: 0.5700 | Accuracy: 0.6190 | F1: 0.5114 | AUROC: 0.6409
Subject 017 | Balanced Accuracy: 0.6518 | Accuracy: 0.5278 | F1: 0.5185 | AUROC: 0.4978
Subject 021 | Balanced Accuracy: 0.5387 | Accuracy: 0.4883 | F1: 0.4263 | AUROC: 0.5341
Subject 022 | Balanced Accuracy: 0.5235 | Accuracy: 0.5263 | F1: 0.5210 | AUROC: 0.5417
Subject 023 | Balanced Accuracy: 0.5148 | Accuracy: 0.6200 | F1: 0.5062 | AUROC: 0.4506
Subject 024 | Balanced Accuracy: 0.5709 | Accuracy: 0.5362 | F1: 0.5328 | AU

In [34]:
#10Fold CV
knn_pn_10f = knn_ten_fold_cv_loop(X_pn, y_pn, knn_pn_params_10f, "KNN", "Positive vs. Negative")


=== 10-Fold CV - KNN (Positive vs. Negative) ===
Accuracy:          0.6447 ± 0.0269
F1:                0.6343 ± 0.0261
Balanced Accuracy: 0.6351 ± 0.0254
AUROC:             0.7058 ± 0.0296
